In [6]:
import pandas as pd
import os

# Read the data file
data_file = pd.read_csv("../../data/GPT-OSS_stepwise/h_prompts_pre_penultimate_sum.csv")

# Read the clean and intervened results
clean_results = pd.read_csv("../../experiments/activation_intervention/output/GPT-OSS_stepwise/frozen_attention_intervened_pattern/non_intervened_pattern_h_pre_penultimate_sum_.csv")
intervened_results = pd.read_csv("../../experiments/activation_intervention/output/GPT-OSS_stepwise/frozen_attention_intervened_pattern/h_pre_penultimate_sum_.csv")

# List to store rows that meet the criteria
filtered_rows = []

# Process each row in the data file
for _, data_row in data_file.iterrows():
    # Find matching rows in clean results
    clean_matches = clean_results[
        (clean_results['base_1_num'] == data_row['base_1_num']) &
        (clean_results['base_2_num'] == data_row['base_2_num']) &
        (clean_results['source_1_num'] == data_row['source_1_num']) &
        (clean_results['source_2_num'] == data_row['source_2_num'])
    ]
    
    # Find matching rows in intervened results
    intervened_matches = intervened_results[
        (intervened_results['base_1_num'] == data_row['base_1_num']) &
        (intervened_results['base_2_num'] == data_row['base_2_num']) &
        (intervened_results['source_1_num'] == data_row['source_1_num']) &
        (intervened_results['source_2_num'] == data_row['source_2_num'])
    ]
    
    # Check if we have exactly 20 matches in each file
    if len(clean_matches) == 20 and len(intervened_matches) == 20:
        # Count how many rows have generated_text equal to counterfactual_output
        clean_count = (clean_matches['generated_text'] == clean_matches['counterfactual_output']).sum()
        intervened_count = (intervened_matches['generated_text'] == intervened_matches['counterfactual_output']).sum()
        
        # Check if clean_count is at least 10 more than intervened_count
        if clean_count >= intervened_count + 14:
            filtered_rows.append(data_row)
    else:
        print(f"Skipping row {data_row['base_1_num']}+{data_row['base_2_num']}, {data_row['source_1_num']}+{data_row['source_2_num']}")

# Create DataFrame from filtered rows
filtered_df = pd.DataFrame(filtered_rows)

# Create output directory if it doesn't exist
output_dir = "../../experiments/steering/output/GPT-OSS_stepwise/steering_dataset"
os.makedirs(output_dir, exist_ok=True)

# Save the filtered dataset
output_path = os.path.join(output_dir, "filtered_steering_dataset.csv")
filtered_df.to_csv(output_path, index=False)

print(f"Filtered dataset saved to {output_path}")
print(f"Original dataset size: {len(data_file)}")
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset saved to ../../experiments/steering/output/GPT-OSS_stepwise/steering_dataset/filtered_steering_dataset.csv
Original dataset size: 256
Filtered dataset size: 120
